# AI Programming — RNN Lab
## IMDB 감성 분류: Embedding + LSTM

이번 실습에서는 **IMDB 영화 리뷰**를 입력받아
리뷰가 긍정(Positive)인지 부정(Negative)인지 분류합니다.

RNN 계열은 현재 NLP의 주류 모델은 아니지만,
**시퀀스를 순서대로 처리하고 hidden state에 정보를 누적한다는 기본 개념**을 이해하기 위해
간단한 LSTM 실습 하나만 수행합니다.

### 학습 목표

- 텍스트가 token ID의 sequence로 표현되는 방식을 확인합니다.
- 길이가 다른 sequence를 padding하여 같은 길이로 맞춥니다.
- `Embedding` layer가 token ID를 dense vector로 변환하는 역할을 이해합니다.
- LSTM이 sequence를 순서대로 처리하는 방식을 확인합니다.
- 마지막 hidden state를 이용한 binary sentiment classification을 구현합니다.
- Training / validation loss와 test accuracy를 확인합니다.

### 전체 흐름

```text
Movie Review
    ↓
Token IDs
    ↓
Padding
    ↓
Embedding
    ↓
LSTM
    ↓
Dense + Sigmoid
    ↓
Positive / Negative
```

> 이번 실습에서는 Attention이나 복잡한 RNN 구조는 다루지 않습니다.  
> **Embedding → LSTM → Classification**의 기본 흐름에만 집중합니다.

## 1. 라이브러리와 기본 설정

- `VOCAB_SIZE = 10,000`: 가장 자주 등장하는 10,000개 단어만 사용
- `MAX_LEN = 200`: 리뷰 길이를 최대 200 token으로 통일
- `EMBED_DIM = 32`: 각 token을 32차원 vector로 표현
- `LSTM_UNITS = 32`: LSTM hidden state의 차원

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input, Embedding, LSTM, Dense
)

tf.keras.utils.set_random_seed(42)

VOCAB_SIZE = 10_000
MAX_LEN = 200
EMBED_DIM = 32
LSTM_UNITS = 32

## 2. IMDB Dataset 불러오기

Keras의 IMDB dataset은 이미 각 단어가 **정수 token ID**로 변환되어 있습니다.

- Training samples: 25,000
- Test samples: 25,000
- `0`: Negative
- `1`: Positive

즉, 이번 실습에서는 raw text tokenization 자체보다
**token ID sequence가 neural network에 어떻게 입력되는지**에 집중합니다.

In [ ]:
(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=VOCAB_SIZE
)

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))
print("First label:", y_train[0])
print("First review length:", len(X_train[0]))
print("First 20 word indices:", X_train[0][:20])

### 확인할 내용

첫 번째 리뷰는 하나의 정수가 아니라 **정수들의 sequence**입니다.

```text
[1, 14, 22, 16, ...]
```

각 숫자는 단어의 크기나 의미를 나타내는 값이 아니라
**vocabulary에서 해당 token을 가리키는 ID**입니다.

## 3. Sequence 길이 맞추기

리뷰마다 token 수가 다르기 때문에 그대로 batch를 만들기 어렵습니다.

`pad_sequences()`를 이용해 모든 리뷰를 길이 200으로 맞춥니다.

```text
짧은 리뷰 → 뒤에 0을 추가
긴 리뷰   → 200 token 이후를 잘라냄
```

최종 input shape는 다음과 같습니다.

```text
(samples, 200)
```

In [ ]:
X_train = pad_sequences(
    X_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 4. Basic LSTM Classifier

모델은 매우 단순하게 구성합니다.

```text
Token IDs
   ↓
Embedding
   ↓
LSTM
   ↓
Dense(1, sigmoid)
```

### Embedding

`Embedding(VOCAB_SIZE, 32)`는 각 token ID를 32차원의 dense vector로 변환합니다.

입력 shape:

```text
(batch, 200)
```

Embedding 출력 shape:

```text
(batch, 200, 32)
```

### LSTM

`LSTM(32)`는 sequence를 순서대로 처리하고,
기본 설정에서는 **마지막 hidden state 하나만 출력**합니다.

출력 shape:

```text
(batch, 32)
```

마지막 `Dense(1, sigmoid)`는 리뷰가 Positive일 probability를 출력합니다.

In [ ]:
lstm_model = Sequential([
    Input(shape=(MAX_LEN,)),
    Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True),
    LSTM(LSTM_UNITS),
    Dense(1, activation="sigmoid")
])

lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

lstm_model.summary()

### ✅ 체크포인트

다음 세 가지를 연결해서 이해하세요.

```text
Embedding
→ 각 token의 표현

LSTM
→ sequence 전체의 정보를 누적

Dense + Sigmoid
→ 리뷰 전체를 Positive / Negative로 분류
```

## 5. Model Training

Training data의 20%를 validation set으로 사용합니다.

Validation loss가 더 이상 좋아지지 않으면
`EarlyStopping`이 학습을 종료하고 가장 좋은 weight를 복원합니다.

수업에서는 epoch 수 자체보다 **training loss와 validation loss가 어떻게 변하는지**를 확인하세요.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_lstm = lstm_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

## 6. Test Set 평가

Training이나 model selection에 사용하지 않은 test set으로
최종 성능을 확인합니다.

In [ ]:
lstm_loss, lstm_acc = lstm_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"Test loss: {lstm_loss:.4f}")
print(f"Test accuracy: {lstm_acc:.4f}")

## 7. Learning Curve

Training loss와 validation loss를 함께 확인합니다.

### 확인할 내용

- 두 loss가 함께 감소하는가?
- 어느 시점부터 validation loss가 더 이상 좋아지지 않는가?
- Early stopping이 너무 늦거나 너무 일찍 발생하지는 않는가?

In [ ]:
plt.plot(history_lstm.history["loss"], label="Training Loss")
plt.plot(history_lstm.history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

## 8. Review 하나의 Prediction 확인

Test review 하나를 선택하여

- 실제 label
- model prediction
- Positive probability

를 확인합니다.

In [ ]:
sample_id = 0
sample = X_test[sample_id:sample_id + 1]

probability = lstm_model.predict(
    sample,
    verbose=0
)[0, 0]

prediction = int(probability >= 0.5)

print(
    "True label:",
    "Positive" if y_test[sample_id] == 1 else "Negative"
)

print(
    "Prediction:",
    "Positive" if prediction == 1 else "Negative"
)

print(f"Positive probability: {probability:.4f}")

## 9. 직접 해보기

다음 중 한두 가지만 바꾸어 결과를 비교해 보세요.

1. `MAX_LEN = 100`과 `200` 비교
2. `EMBED_DIM = 16`, `32`, `64` 비교
3. `LSTM_UNITS = 16`, `32`, `64` 비교
4. Test review의 `sample_id`를 바꾸어 예측 확인

### 생각해 보기

- `MAX_LEN`이 너무 짧으면 어떤 정보가 사라질까요?
- `LSTM_UNITS`를 크게 하면 parameter 수는 어떻게 변할까요?
- 마지막 hidden state 하나만으로 긴 리뷰 전체를 표현하는 데 어떤 한계가 있을까요?

# 10. 정리

이번 실습에서는 RNN 계열 모델의 가장 기본적인 text-classification pipeline만 확인했습니다.

```text
Token IDs
   ↓
Embedding
   ↓
LSTM
   ↓
Final Hidden State
   ↓
Sigmoid
   ↓
Sentiment
```

### 꼭 기억할 것

1. **Token ID는 숫자의 크기에 의미가 있는 값이 아니라 vocabulary index입니다.**
2. **Embedding은 token ID를 학습 가능한 dense representation으로 변환합니다.**
3. **LSTM은 token을 순서대로 처리하면서 hidden state를 갱신합니다.**
4. **`LSTM(32)`는 기본적으로 마지막 hidden state를 반환합니다.**
5. **Binary sentiment classification에서는 `Dense(1, sigmoid)`를 사용할 수 있습니다.**

RNN/LSTM은 Transformer 이전의 대표적인 sequence model이며,
이번 실습의 목적은 현대 NLP 성능 경쟁보다는
**sequence modeling의 기본 개념을 코드로 확인하는 것**입니다.